# Quick start: the course training workflow from a notebook

This notebook does what `scripts/train.py` does, step by step. Run it with the **DL 2027 (Python 3.12)** kernel after `pip install -e .` in the repository folder.

In [ ]:
from pathlib import Path
import torch
from dlcourse.config import load_config
from dlcourse.reproducibility import set_seed, get_device

cfg = load_config(Path('..') / 'configs' / 'default.yaml', ['train.epochs=5', 'output.runs_dir=../runs', 'name=notebook-demo'])
set_seed(cfg['seed'], cfg['deterministic'])
device = get_device(cfg['device'])
print('device:', device, '| seed:', cfg['seed'])

## Data: seeded train / validation / test split
The split depends only on the seed. The test set stays untouched until the very end.

In [ ]:
from dlcourse.data import load_splits, make_loaders, dataset_info
train, val, test = load_splits(cfg)
loaders = make_loaders(train, val, test, cfg)
print(len(train), len(val), len(test))

## Model, optimizer, training

In [ ]:
from dlcourse.models import build_model, build_optimizer, count_parameters
from dlcourse.training import fit
info = dataset_info(cfg['data']['dataset'])
model = build_model(cfg, info['in_features'], info['n_classes']).to(device)
opt = build_optimizer(cfg, model.parameters())
print('parameters:', count_parameters(model))
run_dir = Path(cfg['output']['runs_dir']) / cfg['name'] / f"seed{cfg['seed']}"
run_dir.mkdir(parents=True, exist_ok=True)
history = fit(model, loaders, opt, cfg, run_dir, device)

## Evaluate the best-validation checkpoint on the test set (once)

In [ ]:
from dlcourse.training import load_weights, run_epoch
load_weights(model, run_dir / 'best.pt', device)
test_loss, test_acc = run_epoch(model, loaders['test'], device)
print(f'test loss {test_loss:.4f} | test accuracy {test_acc:.3f}')

## Learning curves

In [ ]:
import pandas as pd
df = pd.read_csv(run_dir / 'metrics.csv')
ax = df.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', title='Loss')
ax.set_ylabel('cross-entropy');

**Try it:** rerun with `seed=2` and `seed=3` (edit the override list in the first cell). How much does the test accuracy vary? That spread is why the course asks for results across several seeds.